# Databricks Lakehouse Data Pipeline

Build a Medallion Architecture pipeline using PySpark, Delta Lake, and Spark SQL.

- Bronze: ingest the source retail data with minimal changes.
- Silver: clean and standardize the data.
- Gold: create business summaries for analysis.
- Delta Lake: inspect table history and demonstrate time travel.

Compute: Databricks Free Edition serverless.

In [0]:
from pyspark.sql import functions as F

source_path = "/Volumes/workspace/default/retail_pipeline_files/online_retail.csv"
bronze_table = "workspace.default.bronze_retail_transactions"

raw_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(source_path)
)

bronze_df = (
    raw_df
    .withColumn("_source_file", F.lit("online_retail.csv"))
    .withColumn("_ingested_at", F.current_timestamp())
)

(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(bronze_table)
)

print(f"Bronze table created: {bronze_table}")
display(spark.table(bronze_table).limit(5))

Bronze table created: workspace.default.bronze_retail_transactions


InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,_source_file,_ingested_at
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,online_retail.csv,2026-11-13T03:11:22.429Z
536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,online_retail.csv,2026-11-13T03:11:22.429Z
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,online_retail.csv,2026-11-13T03:11:22.429Z
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,online_retail.csv,2026-11-13T03:11:22.429Z
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,online_retail.csv,2026-11-13T03:11:22.429Z


In [0]:
from pyspark.sql import functions as F

bronze_table = "workspace.default.bronze_retail_transactions"
silver_table = "workspace.default.silver_retail_transactions"

silver_df = (
    spark.table(bronze_table)
    .dropDuplicates()
    .withColumn("InvoiceNo", F.col("InvoiceNo").cast("string"))
    .withColumn("StockCode", F.col("StockCode").cast("string"))
    .withColumn("Description", F.col("Description").cast("string"))
    .withColumn("Quantity", F.col("Quantity").cast("int"))
    .withColumn(
        "InvoiceDate",
        F.to_timestamp("InvoiceDate", "yyyy-MM-dd HH:mm:ss")
    )
    .withColumn("UnitPrice", F.col("UnitPrice").cast("double"))
    .withColumn("CustomerID", F.col("CustomerID").cast("double"))
    .withColumn("LineTotal", F.col("Quantity") * F.col("UnitPrice"))
    .withColumn(
        "TransactionType",
        F.when(
            F.lower(F.trim(F.col("Description"))) == "adjust bad debt",
            "financial_adjustment"
        )
        .when(F.col("InvoiceNo").startswith("C"), "cancellation")
        .when(
            (F.col("Quantity") < 0) & (F.col("UnitPrice") == 0),
            "unpriced_negative_quantity"
        )
        .when(
            (F.col("Quantity") < 0) & (F.col("UnitPrice") > 0),
            "return"
        )
        .otherwise("sale")
    )
    .withColumn("HasMissingCustomerID", F.col("CustomerID").isNull())
    .withColumn("HasMissingDescription", F.col("Description").isNull())
    .withColumn("HasZeroUnitPrice", F.col("UnitPrice") == 0)
)

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

print(f"Silver table created: {silver_table}")
print(f"Rows after exact deduplication: {spark.table(silver_table).count():,}")
display(spark.table(silver_table).limit(5))

Silver table created: workspace.default.silver_retail_transactions
Rows after exact deduplication: 536,641


InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,_source_file,_ingested_at,LineTotal,TransactionType,HasMissingCustomerID,HasMissingDescription,HasZeroUnitPrice
536369,21756,BATH BUILDING BLOCK WORD,3,2010-12-01T08:35:00.000Z,5.95,13047.0,United Kingdom,online_retail.csv,2026-11-13T03:11:22.429Z,17.85,sale,false,false,false
536370,22629,SPACEBOY LUNCH BOX,24,2010-12-01T08:45:00.000Z,1.95,12583.0,France,online_retail.csv,2026-11-13T03:11:22.429Z,46.8,sale,false,false,false
536370,21913,VINTAGE SEASIDE JIGSAW PUZZLES,12,2010-12-01T08:45:00.000Z,3.75,12583.0,France,online_retail.csv,2026-11-13T03:11:22.429Z,45.0,sale,false,false,false
536370,22540,MINI JIGSAW CIRCUS PARADE,24,2010-12-01T08:45:00.000Z,0.42,12583.0,France,online_retail.csv,2026-11-13T03:11:22.429Z,10.08,sale,false,false,false
536376,22114,HOT WATER BOTTLE TEA AND SYMPATHY,48,2010-12-01T09:32:00.000Z,3.45,15291.0,United Kingdom,online_retail.csv,2026-11-13T03:11:22.429Z,165.60000000000002,sale,false,false,false


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.gold_monthly_sales
USING DELTA
AS
SELECT
    DATE_TRUNC('MONTH', InvoiceDate) AS sales_month,
    COUNT(DISTINCT CASE
        WHEN TransactionType = 'sale' THEN InvoiceNo
    END) AS sales_invoice_count,
    COUNT(DISTINCT CASE
        WHEN TransactionType = 'sale' THEN CustomerID
    END) AS known_customer_count,
    ROUND(SUM(CASE
        WHEN TransactionType = 'sale' THEN LineTotal ELSE 0
    END), 2) AS gross_sales,
    ROUND(SUM(CASE
        WHEN TransactionType IN ('sale', 'cancellation', 'return')
        THEN LineTotal ELSE 0
    END), 2) AS net_sales
FROM workspace.default.silver_retail_transactions
WHERE TransactionType IN ('sale', 'cancellation', 'return')
GROUP BY DATE_TRUNC('MONTH', InvoiceDate);

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM workspace.default.gold_monthly_sales
ORDER BY sales_month;

sales_month,sales_invoice_count,known_customer_count,gross_sales,net_sales
2010-12-01T00:00:00.000Z,1629,885,821452.73,746723.61
2011-01-01T00:00:00.000Z,1120,741,689811.61,558448.56
2011-02-01T00:00:00.000Z,1126,758,522545.56,497026.41
2011-03-01T00:00:00.000Z,1531,974,716215.26,682013.98
2011-04-01T00:00:00.000Z,1318,856,536968.49,492367.84
2011-05-01T00:00:00.000Z,1731,1056,769296.61,722094.1
2011-06-01T00:00:00.000Z,1576,991,760547.01,689977.23
2011-07-01T00:00:00.000Z,1540,949,718076.12,680156.99
2011-08-01T00:00:00.000Z,1406,935,746779.32,692448.52
2011-09-01T00:00:00.000Z,1896,1266,1056435.19,1017596.68


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.gold_country_sales
USING DELTA
AS
SELECT
    Country AS country,
    COUNT(DISTINCT CASE
        WHEN TransactionType = 'sale' THEN InvoiceNo
    END) AS sales_invoice_count,
    COUNT(DISTINCT CASE
        WHEN TransactionType = 'sale' THEN CustomerID
    END) AS known_customer_count,
    ROUND(SUM(CASE
        WHEN TransactionType = 'sale' THEN LineTotal ELSE 0
    END), 2) AS gross_sales,
    ROUND(SUM(CASE
        WHEN TransactionType IN ('sale', 'cancellation', 'return')
        THEN LineTotal ELSE 0
    END), 2) AS net_sales
FROM workspace.default.silver_retail_transactions
WHERE TransactionType IN ('sale', 'cancellation', 'return')
GROUP BY Country;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM workspace.default.gold_country_sales
ORDER BY net_sales DESC
LIMIT 10;

country,sales_invoice_count,known_customer_count,gross_sales,net_sales
United Kingdom,18783,3921,8990682.03,8178190.24
Netherlands,95,9,285446.34,284661.54
EIRE,288,3,283140.52,262993.38
Germany,457,94,228678.4,221509.47
France,392,87,209625.37,197317.11
Australia,57,9,138453.81,137009.77
Switzerland,54,21,57067.6,56363.05
Spain,90,30,61558.56,54756.03
Belgium,98,25,41196.34,40910.96
Sweden,36,8,38367.83,36585.41


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.gold_product_sales
USING DELTA
AS
WITH product_sales AS (
    SELECT
        StockCode AS stockcode,
        ROUND(SUM(LineTotal), 2) AS net_sales
    FROM workspace.default.silver_retail_transactions
    WHERE TransactionType IN ('sale', 'cancellation', 'return')
      AND StockCode NOT IN ('DOT', 'POST', '23084')
    GROUP BY StockCode
),
description_counts AS (
    SELECT
        StockCode AS stockcode,
        Description AS product_description,
        COUNT(*) AS description_rows
    FROM workspace.default.silver_retail_transactions
    WHERE TransactionType = 'sale'
      AND StockCode NOT IN ('DOT', 'POST', '23084')
      AND Description IS NOT NULL
      AND TRIM(Description) <> ''
    GROUP BY StockCode, Description
),
ranked_descriptions AS (
    SELECT
        stockcode,
        product_description,
        ROW_NUMBER() OVER (
            PARTITION BY stockcode
            ORDER BY description_rows DESC, product_description
        ) AS description_rank
    FROM description_counts
)
SELECT
    sales.stockcode,
    descriptions.product_description,
    sales.net_sales
FROM product_sales AS sales
LEFT JOIN ranked_descriptions AS descriptions
    ON sales.stockcode = descriptions.stockcode
   AND descriptions.description_rank = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT stockcode, product_description, net_sales
FROM workspace.default.gold_product_sales
ORDER BY net_sales DESC
LIMIT 10;

stockcode,product_description,net_sales
22423,REGENCY CAKESTAND 3 TIER,164459.49
47566,PARTY BUNTING,98243.88
85123A,WHITE HANGING HEART T-LIGHT HOLDER,97838.45
85099B,JUMBO BAG RED RETROSPOT,92175.79
22086,PAPER CHAIN KIT 50'S CHRISTMAS,63715.24
84879,ASSORTED COLOUR BIRD ORNAMENT,58792.42
79321,CHILLI LIGHTS,53746.66
22502,PICNIC BASKET WICKER SMALL,51023.52
22197,POPCORN HOLDER,50967.92
23298,SPOTTY BUNTING,42665.37


In [0]:
%sql
DESCRIBE HISTORY workspace.default.silver_retail_transactions;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-11-13T03:14:11.000Z,75976966603071,nagulanjp2003@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1795977480208633),7f4d0a90-65d1-4d61-a03a-db67a2959d76,1113-031056-j4su71rl-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 536641, numOutputBytes -> 5385670)",null,Databricks-Runtime/19.14.x-aarch64-photon-scala2.13


In [0]:
%sql
SELECT COUNT(*) AS descriptions_needing_trim
FROM workspace.default.silver_retail_transactions
WHERE Description IS NOT NULL
  AND Description <> TRIM(Description);

descriptions_needing_trim
111983


In [0]:
%sql
UPDATE workspace.default.silver_retail_transactions
SET
    Description = TRIM(Description),
    HasMissingDescription = CASE
        WHEN TRIM(Description) = '' THEN true
        ELSE false
    END
WHERE Description IS NOT NULL
  AND Description <> TRIM(Description);

num_affected_rows
111983


In [0]:
%sql
DESCRIBE HISTORY workspace.default.silver_retail_transactions;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-11-13T07:10:15.000Z,75976966603071,nagulanjp2003@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1795977480208633),35ba6ae2-800a-4ed4-8187-56c665ed26f0,1113-070946-hn5cg1e6-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 6642936, p25FileSize -> 5768455, numDeletionVectorsRemoved -> 1, minFileSize -> 5768455, numAddedFiles -> 1, maxFileSize -> 5768455, p75FileSize -> 5768455, p50FileSize -> 5768455, numAddedBytes -> 5768455)",null,Databricks-Runtime/19.14.x-aarch64-photon-scala2.13
1,2026-11-13T07:10:12.000Z,75976966603071,nagulanjp2003@gmail.com,UPDATE,"Map(predicate -> [""(isnotnull(Description#10136) AND NOT (Description#10136 = trim(Description#10136, None)))""])",null,List(1795977480208633),35ba6ae2-800a-4ed4-8187-56c665ed26f0,1113-070946-hn5cg1e6-v2n,0,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 12523, numDeletionVectorsUpdated -> 0, scanTimeMs -> 8999, numAddedFiles -> 1, numUpdatedRows -> 111983, numAddedBytes -> 1257266, rewriteTimeMs -> 3483)",null,Databricks-Runtime/19.14.x-aarch64-photon-scala2.13
0,2026-11-13T03:14:11.000Z,75976966603071,nagulanjp2003@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1795977480208633),7f4d0a90-65d1-4d61-a03a-db67a2959d76,1113-031056-j4su71rl-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 536641, numOutputBytes -> 5385670)",null,Databricks-Runtime/19.14.x-aarch64-photon-scala2.13


In [0]:
%sql
SELECT
    'version_0_before_trim' AS snapshot,
    COUNT(*) AS row_count,
    SUM(CASE
        WHEN Description IS NOT NULL AND Description <> TRIM(Description)
        THEN 1 ELSE 0
    END) AS descriptions_needing_trim
FROM workspace.default.silver_retail_transactions VERSION AS OF 0

UNION ALL

SELECT
    'current_after_trim' AS snapshot,
    COUNT(*) AS row_count,
    SUM(CASE
        WHEN Description IS NOT NULL AND Description <> TRIM(Description)
        THEN 1 ELSE 0
    END) AS descriptions_needing_trim
FROM workspace.default.silver_retail_transactions;

snapshot,row_count,descriptions_needing_trim
version_0_before_trim,536641,111983
current_after_trim,536641,0


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.gold_product_sales AS
WITH product_sales AS (
    SELECT
        StockCode AS stockcode,
        ROUND(SUM(LineTotal), 2) AS net_sales
    FROM workspace.default.silver_retail_transactions
    WHERE TransactionType IN ('sale', 'cancellation', 'return')
      AND StockCode NOT IN ('DOT', 'POST', '23084')
    GROUP BY StockCode
),
description_counts AS (
    SELECT
        StockCode AS stockcode,
        TRIM(Description) AS product_description,
        COUNT(*) AS description_count,
        ROW_NUMBER() OVER (
            PARTITION BY StockCode
            ORDER BY COUNT(*) DESC, TRIM(Description)
        ) AS description_rank
    FROM workspace.default.silver_retail_transactions
    WHERE TransactionType = 'sale'
      AND Description IS NOT NULL
      AND TRIM(Description) <> ''
      AND StockCode NOT IN ('DOT', 'POST', '23084')
    GROUP BY StockCode, TRIM(Description)
)
SELECT
    sales.stockcode,
    descriptions.product_description,
    sales.net_sales
FROM product_sales AS sales
LEFT JOIN description_counts AS descriptions
    ON sales.stockcode = descriptions.stockcode
   AND descriptions.description_rank = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT stockcode, product_description, net_sales
FROM workspace.default.gold_product_sales
ORDER BY net_sales DESC
LIMIT 10;

stockcode,product_description,net_sales
22423,REGENCY CAKESTAND 3 TIER,164459.49
47566,PARTY BUNTING,98243.88
85123A,WHITE HANGING HEART T-LIGHT HOLDER,97838.45
85099B,JUMBO BAG RED RETROSPOT,92175.79
22086,PAPER CHAIN KIT 50'S CHRISTMAS,63715.24
84879,ASSORTED COLOUR BIRD ORNAMENT,58792.42
79321,CHILLI LIGHTS,53746.66
22502,PICNIC BASKET WICKER SMALL,51023.52
22197,POPCORN HOLDER,50967.92
23298,SPOTTY BUNTING,42665.37


## Pipeline results

- **Bronze:** Ingested 541,909 source rows from CSV into a Delta table.
- **Silver:** Removed 5,268 exact duplicates, leaving 536,641 rows; standardized data types and trimmed descriptions.
- **Gold:** Created monthly sales, country sales, and product sales summary tables.
- **Delta Lake:** Recorded table history and compared version 0 with the current table. Both had 536,641 rows; padded descriptions went from 111,983 to 0.
- **Compute:** Databricks Free Edition serverless compute.